<h1 style="color: #008080;">Scraper del Observatorio de Compra Pública de Costa Rica.</h1>

Descarga los archivos ZIP mensuales del repositorio público del Observatorio de Compra Pública de Costa Rica (en Azure Blob Storage)
y extrae los datos de adjudicaciones en un único dataframe y archivo CSV consolidado.

URL sitio web:
    https://www.observatoriocomprapublica.go.cr/descargas-sicop/
    
URL base de descarga:
    https://dlsaobservatorioprod.blob.core.windows.net/fs-synapse-observatorio-produccion/Zip/yyyymm.zip

Instalación de paquetes:
`pip install requests`
`pip install pandas`

Importación de las librerías necesarias para obtener la información y crear el data set.

In [19]:
import pandas as pd
import requests

import io
import zipfile
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

Se deben definir la cantidad de meses de datos por consultar en la dirección de la página web para crear el data set.

In [33]:
BASE_URL = "https://dlsaobservatorioprod.blob.core.windows.net/fs-synapse-observatorio-produccion/Zip"
MESES_A_DESCARGAR = 24 #Número de meses a descargar, incluyendo el mes actual (24 por defecto para obtener 2 años de datos).
#12 meses puede tardar unos 7 minutos en descargar con una velocidad de 10Mbps. 24 meses puede tardar unos 15 minutos.
ARCHIVO_CSV = "ProcedimientoAdjudicacion.csv"

Definimos las columnas que son de interés para el análisis de datos sobre licitaciones públicas del archivo `"ProcedimientoAdjudicacion.csv"`.

In [27]:
COLUMNAS_DF = [
    "CEDULA",
    "INSTITUCION",
    "ANO",
    "NUMERO_PROCEDIMIENTO",
    "DESCR_PROCEDIMIENTO",
    "NRO_SICOP",
    "TIPO_PROCEDIMIENTO",
    "MODALIDAD_PROCEDIMIENTO",
    "NOMBRE_PROVEEDOR",
    "MONTO_ADJU_LINEA_CRC",
    "MONTO_ADJU_LINEA_USD",
    "FECHA_ADJUD_FIRME",
    "PROD_ID",
    "DESCR_BIEN_SERVICIO",
    "CANTIDAD"
]

COLUMNAS_NO_DF = [ #COLUMNAS QUE NO QUEREMOS INCLUIR EN EL DATAFRAME, ya que no aportan información relevante para el análisis o tienen muchos valores faltantes.
    "LINEA",
    "UNIDAD_MEDIDA",
    "MONTO_UNITARIO",
    "MONEDA_PRECIO_EST",
    "MONEDA_ADJUDICADA",
    "MONTO_ADJU_LINEA",
    "FECHA_SOL_CONTRA",
    "PERFIL_PROV",
    "CEDULA_PROVEEDOR",
    "CEDULA_REPRESENTANTE",
    "REPRESENTANTE",
    "OBJETO_GASTO",
    "fecha_rev",
    "FECHA_SOL_CONTRA_CL",
    "PROD_ID_CL"
]

Función para crear una lista con los meses por consultar de información.
Cada respuesta de la página web entregará los datos correspondientes un mes específico de licitaciones.

El código actual ignora el mes actual para entregar una lista de datos historicos comenzando con el mes anterior.

In [34]:
def meses_recientes(n: int) -> list[str]:
    #Genera lista de códigos yyyymm para los últimos n meses.
    hoy = date.today()
    meses = []
    for i in range(n):
        d = hoy - relativedelta(months= (i+1)) #Restamos i+1 meses para no incluir el mes actual. Si desea incluir el mes actual, use i en lugar de i+1.
        meses.append(d.strftime("%Y%m"))
    return meses

meses = meses_recientes(MESES_A_DESCARGAR)
print("Meses a descargar:", meses)

Meses a descargar: ['202604', '202603', '202602', '202601', '202512', '202511', '202510', '202509', '202508', '202507', '202506', '202505', '202504', '202503', '202502', '202501', '202412', '202411', '202410', '202409', '202408', '202407', '202406', '202405']


Función `descargar_zip` para solicitar archivos zip a la página web del observatorio.

Luego de solicitar los archivos zip, extrae el archivo "ProcedimientoAdjudicacion.csv" en un data frame y se eliminan las columnas no deseadas.

In [ ]:
def descargar_zip(yyyymm: str) -> zipfile.ZipFile | None:
    url = f"{BASE_URL}/{yyyymm}.zip"
    try:
        r = requests.get(url, timeout=120)
        if r.status_code != 200:
            print(f"  No disponible ({r.status_code})")
            return None
        mb = len(r.content) / 1024 / 1024
        print(f"{yyyymm}  Descargado ({mb:.1f} MB)")
        return zipfile.ZipFile(io.BytesIO(r.content))
    except Exception as e:
        print(f"  Error al descargar {yyyymm}: {e}")
        return None

df_procedimiento_adj = pd.DataFrame()

for yyyymm in reversed(meses):
    #descargar el zip y abrir el csv dentro del zip.
    zip_file = descargar_zip(yyyymm)
    print (zip_file.namelist())
    if zip_file is None:
        print(f" Zip {yyyymm} no disponible.")
        continue

    if ARCHIVO_CSV in zip_file.namelist():
        contenido = zip_file.open(ARCHIVO_CSV).read().decode("utf-8-sig", errors="replace")
    elif (yyyymm+"/"+ARCHIVO_CSV) in zip_file.namelist(): #EN DESARROLLO: algunos archivos del 2024 tienen columnas con formato diferente
        #intentamos con otros nombres de archivo comunes, ya que el formato no es consistente para archivos del 2024 o antes.
        contenido = zip_file.open(yyyymm+"/"+ARCHIVO_CSV).read().decode("utf-8-sig", errors="replace")

    elif (yyyymm+"/"+yyyymm+"/"+ARCHIVO_CSV) in zip_file.namelist(): #EN DESARROLLO: algunos archivos del 2024 tienen columnas con formato diferente
        #intentamos con otros nombres de archivo comunes, ya que el formato no es consistente para archivos del 2024 o antes.
        contenido = zip_file.open(yyyymm+"/"+yyyymm+"/"+ARCHIVO_CSV).read().decode("utf-8-sig", errors="replace")    
    else:
        print(f"  No se encontró el archivo {ARCHIVO_CSV} en el zip {yyyymm}.")

    #Cargar el contenido del csv en un DataFrame, eliminando las columnas no deseadas
    df_mes = pd.read_csv(io.StringIO(contenido), delimiter=";")[COLUMNAS_DF]

    #Unir DataFrames por filas (axis=0) en un DataFrame final, ignorando los índices originales (ignore_index=True)
    df_procedimiento_adj = pd.concat([df_procedimiento_adj, df_mes], ignore_index=True)

#Convertir columnas de fecha a formato datetime, números a float y demás a tipo string para facilitar su manejo.
df_procedimiento_adj['FECHA_ADJUD_FIRME'] = pd.to_datetime(df_procedimiento_adj['FECHA_ADJUD_FIRME'])
df_procedimiento_adj['CEDULA'] = df_procedimiento_adj['CEDULA'].astype(str)
df_procedimiento_adj['INSTITUCION'] = df_procedimiento_adj['INSTITUCION'].astype(str)
df_procedimiento_adj['ANO'] = df_procedimiento_adj['ANO'].astype('Int16')
df_procedimiento_adj['NUMERO_PROCEDIMIENTO'] = df_procedimiento_adj['NUMERO_PROCEDIMIENTO'].astype(str)
df_procedimiento_adj['DESCR_PROCEDIMIENTO'] = df_procedimiento_adj['DESCR_PROCEDIMIENTO'].astype(str)
df_procedimiento_adj['NRO_SICOP'] = df_procedimiento_adj['NRO_SICOP'].astype(str)
df_procedimiento_adj['TIPO_PROCEDIMIENTO'] = df_procedimiento_adj['TIPO_PROCEDIMIENTO'].astype(str)
df_procedimiento_adj['MODALIDAD_PROCEDIMIENTO'] = df_procedimiento_adj['MODALIDAD_PROCEDIMIENTO'].astype(str)
df_procedimiento_adj['NOMBRE_PROVEEDOR'] = df_procedimiento_adj['NOMBRE_PROVEEDOR'].astype(str)
df_procedimiento_adj['MONTO_ADJU_LINEA_CRC'] = df_procedimiento_adj['MONTO_ADJU_LINEA_CRC'].astype('float64')
df_procedimiento_adj['MONTO_ADJU_LINEA_USD'] = df_procedimiento_adj['MONTO_ADJU_LINEA_USD'].astype('float64')
df_procedimiento_adj['PROD_ID'] = df_procedimiento_adj['PROD_ID'].astype(str)
df_procedimiento_adj['DESCR_BIEN_SERVICIO'] = df_procedimiento_adj['DESCR_BIEN_SERVICIO'].astype(str)
df_procedimiento_adj['CANTIDAD'] = df_procedimiento_adj['CANTIDAD'].astype('float64')

202405  Descargado (29.9 MB)
['202405/', '202405/AdjudicacionesFirme.csv', '202405/Contratos.csv', '202405/DetalleCarteles.csv', '202405/DetalleLineaCartel.csv', '202405/FechaPorEtapas.csv', '202405/FuncionariosInhibicion.csv', '202405/Garantias.csv', '202405/InstitucionesRegistradas.csv', '202405/InvitacionProcedimiento.csv', '202405/LineasAdjudicadas.csv', '202405/LineasContratadas.csv', '202405/LineasOfertadas.csv', '202405/LineasRecibidas.csv', '202405/Ofertas.csv', '202405/OrdenPedido.csv', '202405/ProcedimientoAdjudicacion.csv', '202405/ProcedimientoADM.csv', '202405/Proveedores.csv', '202405/ReajustePrecios.csv', '202405/Recepciones.csv', '202405/RecursosObjecion.csv', '202405/Remates.csv', '202405/SancionProveedores.csv', '202405/SistemaEvaluacionOfertas.csv', '202405/Sistemas.csv']
202406  Descargado (11.1 MB)
['202406/', '202406/AdjudicacionesFirme.csv', '202406/Contratos.csv', '202406/DetalleCarteles.csv', '202406/DetalleLineaCartel.csv', '202406/FechaPorEtapas.csv', '202406

In [37]:
df_procedimiento_adj.head(10)

,CEDULA,INSTITUCION,ANO,NUMERO_PROCEDIMIENTO,DESCR_PROCEDIMIENTO,NRO_SICOP,TIPO_PROCEDIMIENTO,MODALIDAD_PROCEDIMIENTO,NOMBRE_PROVEEDOR,MONTO_ADJU_LINEA_CRC,MONTO_ADJU_LINEA_USD,FECHA_ADJUD_FIRME,PROD_ID,DESCR_BIEN_SERVICIO,CANTIDAD
0,3007045617,JUNTA DE PROTECCION SOCIAL,2024,2024PX-000001-0015600001,Contrato de suscripción de pólizas de seguros ...,20240112797,PROCEDIMIENTO POR EXCEPCIÓN,Según demanda,INSTITUTO NACIONAL DE SEGUROS,3.676568e+04,70.58,2024-05-02,8413151492231709,SEGURO DE GARANTÍA DE FIDELIDAD PARA EL MANEJO...,1.0
1,3007317912,SISTEMA NACIONAL DE AREAS DE CONSERVACION,2023,2023XE-000003-0006800001,Alquiler Inmueble Parque Nacional Volcán Turri...,20231108918,PROCEDIMIENTOS ESPECIALES,Según demanda,CAMPOS FERTILES DE TURRIALBA SOCIEDAD ANONIMA,7.458000e+06,14317.80,2024-05-02,8013150292036393,SERVICIO DE ALQUILER DE EDIFICIOS,12.0
2,2100042001,MINISTERIO DE CULTURA Y JUVENTUD,2023,2023LE-000017-0008000001,Restauración de la Casona Hacienda San Luis.,20231212657,LICITACIÓN MENOR,Según demanda,AMERICA INGENIERIA Y ARQUITECTURA SOCIEDAD ANO...,1.997083e+08,387069.15,2024-05-03,9314171392087616,SERVICIO DE RESTAURACION PATRIMONIAL DE MONUME...,1.0
3,3007087654,COMITE CANTONAL DE DEPORTES Y RECREACION DE HE...,2023,2023LD-000083-0027600001,Servicio de mantenimiento para aires acondicio...,20231109033,LICITACIÓN REDUCIDA,Servicios,INGENIEROS CONSULTORES BEKWO SOCIEDAD ANONIMA,1.000000e+05,190.90,2024-05-01,7210151190034359,MANTENIMIENTO DE AIRES ACONDICIONADOS,1.0
4,3007339600,Concejo Municipal de Distrito de Paquera,2024,2024LD-000002-0040500006,CONTRATACION DE SERVICIO POR DEMANDA DE UN TAL...,20240215973,LICITACIÓN REDUCIDA,Según demanda,REPUESTOS JICARALEÑOS R Y Z SOCIEDAD ANONIMA,2.183661e+06,4311.19,2024-05-04,7818150792047429,MANTENIMIENTO CORRECTIVO Y PREVENTIVO MENOR DE...,1.0
5,3014042058,MUNICIPALIDAD DE SAN JOSE,2024,2024PX-000213-0015499999,COMPRA DE REPUESTOS PARA DIFERENTES ACTIVOS MU...,20240318607,PROCEDIMIENTO POR EXCEPCIÓN,Cantidad definida,AUTOCAMIONES DE COSTA RICA AUTO CORI SOCIEDAD ...,2.927840e+05,578.04,2024-05-04,7818150792079524,SERVICIO DE SUMINISTRO DE REPUESTOS Y ACCESOR...,1.0
6,4000042147,Caja Costarricense de Seguro Social,2024,2024LD-000003-0001102634,Materiales Pruebas psicométricas,20240214532,LICITACIÓN REDUCIDA,Cantidad definida,VIVA CON PROPOSITO SOCIEDAD ANONIMA,4.220000e+04,81.79,2024-05-03,1411153392205243,PIN DE CORRECCIÓN POR MEDIO DE INTERNET DE 25 ...,2.0
7,4000042147,Caja Costarricense de Seguro Social,2024,2024LD-000003-0001102272,Compra de Discos Duros de Estado Sólido para C...,20240317280,LICITACIÓN REDUCIDA,Cantidad definida,JI COMPUTERS EQUIPMENT SOCIEDAD ANONIMA,7.304500e+05,1442.12,2024-05-04,4321150792391430,REPUESTOS PARA COMPUTADORA DE ESCRITORIO,35.0
8,3014042061,Municipalidad de Tibás,2024,2024LD-000014-0002800001,produccion del evento traspaso de poderes,20240317814,LICITACIÓN REDUCIDA,Cantidad definida,THREE VISIONS SOCIEDAD DE RESPONSABILIDAD LIMI...,3.450000e+06,6811.32,2024-05-04,8014160792076794,SERVICIO DE ORGANIZADOR Y PRODUCTOR DE EVENTOS...,1.0
9,3014042126,Municipalidad de Siquirres,2023,2023LD-000039-0025700001,COMPRA DE CARNES PARA CECUDI,20231212745,LICITACIÓN REDUCIDA,Según demanda,MARIA PATRICIA QUESADA QUIROS,5.296000e+03,10.26,2024-05-03,5011151592224966,"CARNE DE POLLO, PECHUGAS DESHUESADAS, PRESENTA...",1.0


In [38]:
df_procedimiento_adj.info()

<class 'pandas.DataFrame'>
RangeIndex: 11053 entries, 0 to 11052
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   CEDULA                   11053 non-null  str           
 1   INSTITUCION              11053 non-null  str           
 2   ANO                      11053 non-null  Int16         
 3   NUMERO_PROCEDIMIENTO     11053 non-null  str           
 4   DESCR_PROCEDIMIENTO      11053 non-null  str           
 5   NRO_SICOP                11053 non-null  str           
 6   TIPO_PROCEDIMIENTO       11053 non-null  str           
 7   MODALIDAD_PROCEDIMIENTO  11053 non-null  str           
 8   NOMBRE_PROVEEDOR         11053 non-null  str           
 9   MONTO_ADJU_LINEA_CRC     11053 non-null  float64       
 10  MONTO_ADJU_LINEA_USD     11053 non-null  float64       
 11  FECHA_ADJUD_FIRME        11053 non-null  datetime64[us]
 12  PROD_ID                  11053 non-null  st

Creación de archivo .csv en el directorio donde se ejecute este código.

In [ ]:
#OPCIONAL: Crear un archivo csv en el directorio actual con el DataFrame final, usando ';' como separador, sin índice y codificación utf-8
df_procedimiento_adj.to_csv('Procedimiento_adj.csv', sep=';', index=False, encoding='utf-8')

Otros DataFrames de interés para demás archivos dentro del zip, usando el mismo método de lectura y decodificación, y eliminando las columnas no deseadas si es necesario:

In [ ]:
# Otros DataFrames de interés para demás archivos dentro del zip, usando el mismo método de lectura y decodificación, y eliminando las columnas no deseadas si es necesario.

contenido = zip_file.open("ProcedimientoAdjudicacion.csv").read().decode("utf-8-sig", errors="replace")
df_procedimiento_adj = pd.read_csv(io.StringIO(contenido), delimiter=";")

contenido = zip_file.open("DetalleLineaCartel.csv").read().decode("utf-8-sig", errors="replace")
df_detalle_linea_carteles = pd.read_csv(io.StringIO(contenido), delimiter=";")

contenido = zip_file.open("Proveedores.csv").read().decode("utf-8-sig", errors="replace")
df_proveedores = pd.read_csv(io.StringIO(contenido), delimiter=";")

contenido = zip_file.open("InstitucionesRegistradas.csv").read().decode("utf-8-sig", errors="replace")
df_instituciones_registradas = pd.read_csv(io.StringIO(contenido), delimiter=";")

contenido = zip_file.open("DetalleCarteles.csv").read().decode("utf-8-sig", errors="replace")
df_detalle_carteles = pd.read_csv(io.StringIO(contenido), delimiter=";").drop(columns=["CODIGO_BPIP","COD_EXCEPCION","DES_EXCEPCION","FECHA_MOD"])
df_detalle_carteles['FECHA_PUBLICACION'] = pd.to_datetime(df_detalle_carteles['FECHA_PUBLICACION'])
df_detalle_carteles['FECHAH_APERTURA'] = pd.to_datetime(df_detalle_carteles['FECHAH_APERTURA'])

contenido = zip_file.open("DetalleLineaCartel.csv").read().decode("utf-8-sig", errors="replace")
df_detalle_linea_carteles = pd.read_csv(io.StringIO(contenido), delimiter=";").drop(columns=["NUMERO_LINEA","NUMERO_PARTIDA" ,"TIPO_CAMBIO_CRC"])
